# Synthetic Data Quality Analysis

**Goal:** Evaluate how well each method reproduces the real stroke population — independent of downstream classifier performance.

A method can produce good classifiers but poor-quality synthetic data (or vice versa). This notebook answers: *do the synthetic samples actually look like real stroke patients?*

**Methods evaluated:** SMOTE, CTGAN, TVAE  
**Reference:** Real stroke cases from the training set

**Analyses:**
1. Distribution similarity (numerical + categorical)
2. Statistical tests (KS test, Chi-square)
3. Correlation preservation
4. t-SNE visualisation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.manifold import TSNE
from imblearn.over_sampling import SMOTENC
from scipy.stats import ks_2samp, chi2_contingency

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

DATA_PATH   = Path('../data/healthcare-dataset-stroke-data.csv')
CTGAN_CACHE = Path('../data/synthetic_ctgan.csv')
TVAE_CACHE  = Path('../data/synthetic_tvae.csv')
RANDOM_STATE = 42

NUMERICAL   = ['age', 'avg_glucose_level', 'bmi']
CATEGORICAL = ['hypertension', 'heart_disease', 'gender', 'ever_married',
               'work_type', 'Residence_type', 'smoking_status']

## 1. Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=['id'])
df = df[df['gender'] != 'Other'].copy()
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

train_raw, _ = train_test_split(
    df, test_size=0.2, random_state=RANDOM_STATE, stratify=df['stroke']
)

# Real minority samples — reference for all comparisons
real_minority = train_raw[train_raw['stroke'] == 1].copy()
print(f'Real stroke samples (train): {len(real_minority)}')

In [ ]:
# CTGAN & TVAE — load from cache (generated in notebook 4)
assert CTGAN_CACHE.exists(), 'Run notebook 4 first to generate CTGAN cache'
assert TVAE_CACHE.exists(),  'Run notebook 4 first to generate TVAE cache'

synthetic_ctgan = pd.read_csv(CTGAN_CACHE)
synthetic_tvae  = pd.read_csv(TVAE_CACHE)

# SMOTE — extract newly generated samples
enc = OrdinalEncoder()
df_enc = df.copy()
df_enc[CATEGORICAL] = enc.fit_transform(df[CATEGORICAL])
X = df_enc[NUMERICAL + CATEGORICAL].values
y = df_enc['stroke'].values
cat_indices = list(range(len(NUMERICAL), len(NUMERICAL) + len(CATEGORICAL)))

X_train, _, y_train, _ = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
scaler = StandardScaler()
n = len(NUMERICAL)
X_train_s = np.hstack([scaler.fit_transform(X_train[:, :n]), X_train[:, n:]])

smotenc = SMOTENC(categorical_features=cat_indices, random_state=RANDOM_STATE)
X_res, y_res = smotenc.fit_resample(X_train_s, y_train)

# Newly generated samples are appended after original data
X_smote_new = X_res[len(X_train_s):]
# Inverse scale numericals, decode categoricals
num_orig = scaler.inverse_transform(X_smote_new[:, :n])
cat_orig = enc.inverse_transform(np.round(X_smote_new[:, n:]).astype(int))
smote_cols = pd.DataFrame(num_orig, columns=NUMERICAL)
smote_cat  = pd.DataFrame(cat_orig, columns=CATEGORICAL)
synthetic_smote = pd.concat([smote_cols, smote_cat], axis=1)
synthetic_smote['stroke'] = 1

print(f'CTGAN synthetic: {len(synthetic_ctgan)}')
print(f'TVAE  synthetic: {len(synthetic_tvae)}')
print(f'SMOTE synthetic: {len(synthetic_smote)}')

## 2. Numerical Distributions

Histogram overlay: real stroke samples vs each synthetic method.

In [ ]:
synthetics = {
    'SMOTE': (synthetic_smote, '#2196F3'),
    'CTGAN': (synthetic_ctgan, '#F44336'),
    'TVAE':  (synthetic_tvae,  '#4CAF50'),
}

fig, axes = plt.subplots(3, 3, figsize=(14, 10))

for row, feat in enumerate(NUMERICAL):
    for col, (name, (syn_df, color)) in enumerate(synthetics.items()):
        ax = axes[row, col]
        ax.hist(real_minority[feat], bins=25, alpha=0.6,
                label='Real', color='#9E9E9E', density=True)
        ax.hist(syn_df[feat], bins=25, alpha=0.6,
                label=name, color=color, density=True)
        ax.set_title(f'{name} — {feat}', fontsize=10)
        ax.legend(fontsize=8)

plt.suptitle('Numerical Feature Distributions: Real vs Synthetic', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/quality_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Statistical Tests

### 3.1 KS Test — Numerical Features

The Kolmogorov-Smirnov test checks whether two samples come from the same distribution.
- **p-value > 0.05** → no significant difference → distributions are similar ✓
- **p-value < 0.05** → significant difference → distributions differ ✗

In [ ]:
ks_results = []

for name, (syn_df, _) in synthetics.items():
    for feat in NUMERICAL:
        stat, pval = ks_2samp(real_minority[feat].dropna(), syn_df[feat].dropna())
        ks_results.append({'Method': name, 'Feature': feat,
                            'KS Stat': round(stat, 4), 'p-value': round(pval, 4),
                            'Similar': '✓' if pval > 0.05 else '✗'})

ks_df = pd.DataFrame(ks_results)
print(ks_df.to_string(index=False))

### 3.2 Chi-Square Test — Categorical Features

Chi-square tests whether the category frequencies in synthetic data match the real data.

In [ ]:
chi_results = []

for name, (syn_df, _) in synthetics.items():
    for feat in CATEGORICAL:
        real_counts = real_minority[feat].value_counts()
        syn_counts  = syn_df[feat].value_counts()
        # Align categories
        all_cats = real_counts.index.union(syn_counts.index)
        real_vals = real_counts.reindex(all_cats, fill_value=0).values
        syn_vals  = syn_counts.reindex(all_cats, fill_value=0).values
        # Skip if degenerate
        if real_vals.sum() == 0 or syn_vals.sum() == 0:
            continue
        contingency = np.array([real_vals, syn_vals])
        _, pval, _, _ = chi2_contingency(contingency)
        chi_results.append({'Method': name, 'Feature': feat,
                             'p-value': round(pval, 4),
                             'Similar': '✓' if pval > 0.05 else '✗'})

chi_df = pd.DataFrame(chi_results)
print(chi_df.to_string(index=False))

### 3.3 Test Score Summary

Count how many features pass (p > 0.05) per method — higher is better.

In [ ]:
all_tests = pd.concat([ks_df.rename(columns={'Similar': 'Pass'})[['Method', 'Pass']],
                       chi_df.rename(columns={'Similar': 'Pass'})[['Method', 'Pass']]])

total = len(NUMERICAL) + len(CATEGORICAL)
summary = all_tests.groupby('Method')['Pass'].apply(lambda x: (x == '✓').sum()).reset_index()
summary.columns = ['Method', 'Tests Passed']
summary['Total'] = total
summary['Score'] = summary['Tests Passed'].astype(str) + ' / ' + summary['Total'].astype(str)
print(summary[['Method', 'Score']].to_string(index=False))

## 4. Correlation Preservation

From the EDA we found that the top features correlated with stroke are `age`, `hypertension`, `heart_disease`. Good synthetic data should preserve these correlations.

In [ ]:
# Encode for correlation computation
def encode_df(df_raw):
    d = df_raw.copy()
    d[CATEGORICAL] = OrdinalEncoder().fit_transform(d[CATEGORICAL])
    return d

real_enc = encode_df(real_minority)

corr_data = {'Real': real_enc}
for name, (syn_df, _) in synthetics.items():
    corr_data[name] = encode_df(syn_df)

# Correlation of each feature with every other feature
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, (label, data) in zip(axes, corr_data.items()):
    corr_matrix = data[NUMERICAL + CATEGORICAL].corr()
    sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0,
                vmin=-1, vmax=1, ax=ax, cbar=ax == axes[-1],
                xticklabels=True, yticklabels=True)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', rotation=0, labelsize=7)

plt.suptitle('Feature Correlation Matrix: Real vs Synthetic', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/quality_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. t-SNE Visualisation

t-SNE projects high-dimensional data into 2D for visual inspection.

**Interpretation:** If synthetic samples are good quality, they should **overlap** with real samples in the 2D space — not form separate clusters. Separate clusters mean the synthetic data occupies a different region of the feature space than real data.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

real_enc_arr = encode_df(real_minority)[NUMERICAL + CATEGORICAL].values

for ax, (name, (syn_df, color)) in zip(axes, synthetics.items()):
    syn_enc_arr = encode_df(syn_df)[NUMERICAL + CATEGORICAL].values

    # Use same number of samples for fair comparison
    n_samples = min(len(real_enc_arr), len(syn_enc_arr), 300)
    idx_real = np.random.choice(len(real_enc_arr), n_samples, replace=False)
    idx_syn  = np.random.choice(len(syn_enc_arr),  n_samples, replace=False)

    combined = np.vstack([real_enc_arr[idx_real], syn_enc_arr[idx_syn]])
    labels   = ['Real'] * n_samples + [name] * n_samples

    tsne = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30)
    proj = tsne.fit_transform(combined)

    ax.scatter(proj[:n_samples, 0],  proj[:n_samples, 1],
               alpha=0.5, s=15, color='#9E9E9E', label='Real')
    ax.scatter(proj[n_samples:, 0],  proj[n_samples:, 1],
               alpha=0.5, s=15, color=color, label=name)
    ax.set_title(f't-SNE: Real vs {name}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('t-SNE — Real vs Synthetic Stroke Samples', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/quality_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Quality Summary

In [ ]:
print('=== Synthetic Data Quality Summary ===')
print()

for name, (syn_df, _) in synthetics.items():
    print(f'--- {name} ---')
    for feat in NUMERICAL:
        r_mean, r_std = real_minority[feat].mean(), real_minority[feat].std()
        s_mean, s_std = syn_df[feat].mean(),        syn_df[feat].std()
        print(f'  {feat:<22} Real: {r_mean:.1f} ± {r_std:.1f}  |  {name}: {s_mean:.1f} ± {s_std:.1f}')

    ks_pass  = (ks_df[ks_df['Method']  == name]['Similar'] == '✓').sum()
    chi_pass = (chi_df[chi_df['Method'] == name]['Similar'] == '✓').sum()
    print(f'  KS tests passed:       {ks_pass} / {len(NUMERICAL)}')
    print(f'  Chi-square tests passed: {chi_pass} / {len(CATEGORICAL)}')
    print()